<a href="https://colab.research.google.com/github/rwcitek/TechEx-LangGraph-workshop/blob/main/notebooks/solutions/06_capstone_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Optimization & Iteration

> **Time:** ~6 minutes hands-on (the lecture portion is the slides).
>
> **What you'll do:** pick **ONE** optimization, apply it, re-run the Module 5 eval suite, and compare the new aggregate to the baseline.

The system, dataset, and scorers below are identical to the Module 5 solution. Once you have baseline numbers, you'll edit ONE thing — prompt, workflow, model, or reliability — and measure whether it actually helps.

> 🏁  **Definition of done:** you can name your change, show before/after numbers, and explain *why* the numbers moved.

> *Solution shows worked implementations of all four optimization quadrants. The end-to-end comparison applies Option A, but the others are runnable too — uncomment the `improved_app = build_graph_v2_*` line you want to measure.*

## 1.  Setup

In [1]:
%pip install -q \
    langgraph==0.2.* \
    langchain==0.3.* \
    langchain-openai==0.2.*


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.86 which is incompatible.


In [2]:
import os
from typing import TypedDict, Annotated, Literal
from operator import add
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
import json
import time, random


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
def _ensure_key(name: str, optional: bool = False) -> None:
    """Load an API key from (in order): existing env, Colab Secrets, or getpass.

    Colab Secrets are the recommended path for this workshop — set them ONCE
    via the 🔑 key icon in Colab's left sidebar and every notebook will pick
    them up automatically.
    """
    if os.environ.get(name):
        print(f"  ✓  {name} already set in environment")
        return
    # 1. Try Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✓  {name} loaded from Colab Secrets")
            return
    except Exception:
        pass
    # 2. Fallback — prompt the user
    import getpass
    val = getpass.getpass(f"Paste your {name}{' (optional)' if optional else ''}: ").strip()
    if val:
        os.environ[name] = val
        print(f"  ✓  {name} set")
    elif optional:
        print(f"  •  {name} skipped (optional)")
    else:
        print(f"  ⚠   {name} skipped — you'll hit errors later without it")

_ensure_key("OPENAI_API_KEY")

print("Ready.")


  ✓  OPENAI_API_KEY loaded from Colab Secrets
Ready.


## 2.  Baseline system

In [4]:
KB = {
    "billing": [
        {"id": "B-001", "title": "Billing cycle and prorations",
         "text": "We bill on the same day each month. If you change plans mid-cycle, the next invoice is prorated."},
        {"id": "B-002", "title": "Refund policy",
         "text": "Refunds are available within 14 days. Issued to original payment method, settle in 5 business days."},
        {"id": "B-003", "title": "Failed payments",
         "text": "Failed payments retry once a day for 3 days, then 7-day grace period."},
    ],
    "technical": [
        {"id": "T-001", "title": "Login issues",
         "text": "Clear cookies. For MFA failures check spam and verify phone."},
        {"id": "T-002", "title": "API rate limits",
         "text": "Standard plan: 60 requests per minute. Enterprise: 600. Respect Retry-After header."},
        {"id": "T-003", "title": "Export failures",
         "text": "Retry from the same dialog — export jobs are idempotent."},
    ],
    "account": [
        {"id": "A-001", "title": "Password reset",
         "text": "Use Forgot password. Reset email valid for 30 minutes. Check spam."},
        {"id": "A-002", "title": "Account closure",
         "text": "Settings → Account → Close. 30-day pending-deletion window."},
        {"id": "A-003", "title": "Ownership transfer",
         "text": "Owner invites new owner as Admin, both confirm via email."},
    ],
}


class TriageState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "account"] | None
    urgency:  Literal["low", "med", "high"] | None
    retrieved: list[dict]
    draft: str
    verdict: Literal["pass", "revise"] | None
    revision_count: int
    revisions: Annotated[list[str], add]


def make_initial_state(ticket):
    return {"ticket": ticket, "category": None, "urgency": None,
            "retrieved": [], "draft": "", "verdict": None,
            "revision_count": 0, "revisions": []}


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def parse_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"): text = text[4:]
    return json.loads(text.strip())


# --- Baseline agents (you'll modify ONE of these in the lab) ---

CLASSIFY_PROMPT = """\
Categorize the ticket as one of: billing, technical, account.
Rate urgency as: low, med, high.
Return JSON: {{"category": "...", "urgency": "..."}}

TICKET: {ticket}
"""

def classify(state):
    r = llm.invoke(CLASSIFY_PROMPT.format(ticket=state["ticket"]))
    p = parse_json(r.content)
    return {"category": p["category"], "urgency": p["urgency"]}


def retrieve(state):
    return {"retrieved": KB.get(state["category"], [])}


DRAFT_PROMPT = """\
You are a customer support agent.

Reply to the ticket using ONLY the policies below. If a policy doesn't
answer the question, say so. Do not invent details.

POLICIES:
{context}

TICKET: {ticket}

Write a concise reply (3-6 sentences). Cite the policy ID when used.
"""

def draft(state):
    ctx = "\n\n".join(f"[{d['id']}] {d['title']}\n{d['text']}"
                        for d in state["retrieved"])
    r = llm.invoke(DRAFT_PROMPT.format(context=ctx, ticket=state["ticket"]))
    return {"draft": r.content, "revisions": [r.content]}


QA_PROMPT = """\
Decide if the DRAFT is acceptable. Return JSON: {{"verdict": "pass" | "revise"}}
TICKET: {ticket}
DRAFT:  {draft}
"""

def qa(state):
    r = llm.invoke(QA_PROMPT.format(ticket=state["ticket"], draft=state["draft"]))
    p = parse_json(r.content)
    return {"verdict": p["verdict"], "revision_count": state["revision_count"] + 1}


MAX_REVISIONS = 2

def route_qa(state):
    if state["verdict"] == "pass": return "END"
    if state["revision_count"] >= MAX_REVISIONS: return "END"
    return "drafter"


def build_graph(classify_fn=classify, retrieve_fn=retrieve,
                draft_fn=draft, qa_fn=qa):
    g = StateGraph(TriageState)
    g.add_node("classify", classify_fn)
    g.add_node("retrieve", retrieve_fn)
    g.add_node("drafter",  draft_fn)
    g.add_node("qa",       qa_fn)
    g.set_entry_point("classify")
    g.add_edge("classify", "retrieve")
    g.add_edge("retrieve", "drafter")
    g.add_edge("drafter",  "qa")
    g.add_conditional_edges("qa", route_qa,
        {"drafter": "drafter", "END": END})
    return g.compile()


app = build_graph()
print("Baseline system ready.")

Baseline system ready.


## 3.  Eval suite

In [5]:
EVAL_EXAMPLES = [
    {"id": "E-001", "inputs": {"ticket": "Hi, my monthly bill is $50 higher than last month. Can you check?"},
     "outputs": {"category": "billing", "urgency": "med",
                 "must_mention": ["proration"], "must_cite": ["B-001"]}},
    {"id": "E-002", "inputs": {"ticket": "Charged me twice for the same month. Refund please."},
     "outputs": {"category": "billing", "urgency": "high",
                 "must_mention": ["refund", "14 days"], "must_cite": ["B-002"]}},
    {"id": "E-003", "inputs": {"ticket": "I keep getting 429 errors from your API on the Standard plan."},
     "outputs": {"category": "technical", "urgency": "low",
                 "must_mention": ["60", "Retry-After"], "must_cite": ["T-002"]}},
    {"id": "E-004", "inputs": {"ticket": "Data export failed twice this morning."},
     "outputs": {"category": "technical", "urgency": "med",
                 "must_mention": ["retry", "idempotent"], "must_cite": ["T-003"]}},
    {"id": "E-005", "inputs": {"ticket": "Forgot my password and reset email never came. Demo in 30 minutes!"},
     "outputs": {"category": "account", "urgency": "high",
                 "must_mention": ["spam", "30 minutes"], "must_cite": ["A-001"]}},
    {"id": "E-006", "inputs": {"ticket": "What's the meaning of life? Also can you upgrade my plan to Enterprise?"},
     "outputs": {"category": "billing", "urgency": "low",
                 "must_mention": ["upgrade"], "must_cite": []}},
]


def score_category(run, ex):
    return {"key": "category_match",
            "score": int(run["outputs"]["category"] == ex["outputs"]["category"])}


def score_format(run, ex):
    d = run["outputs"]["draft"]
    leak = any(t in d for t in ["{ticket}", "{context}"])
    miss = [c for c in ex["outputs"].get("must_cite", []) if c not in d]
    return {"key": "format_ok", "score": int(not leak and not miss)}


JUDGE_PROMPT = """\
Score the REPLY from 1 (wrong) to 5 (correct, on-policy, helpful).
Return JSON: {{"score": <int>, "reason": "..."}}

TICKET: {ticket}
SHOULD MENTION: {must_mention}
REPLY: {draft}
"""

def judge_helpfulness(run, ex):
    prompt = JUDGE_PROMPT.format(
        ticket=ex["inputs"]["ticket"],
        must_mention=", ".join(ex["outputs"].get("must_mention", [])),
        draft=run["outputs"]["draft"],
    )
    r = llm.invoke(prompt)
    p = parse_json(r.content)
    return {"key": "helpfulness", "score": p["score"] / 5.0}


SCORERS = [score_category, score_format, judge_helpfulness]


def run_eval(app, examples, scorers):
    rows = []
    for ex in examples:
        state = app.invoke(make_initial_state(ex["inputs"]["ticket"]))
        run = {"outputs": state}
        for s in scorers:
            row = s(run, ex)
            rows.append({"example": ex["id"], **row})
    return rows


def aggregate(rows):
    by = {}
    for r in rows:
        by.setdefault(r["key"], []).append(r["score"])
    return {k: sum(v) / len(v) for k, v in by.items()}


print(f"Loaded {len(EVAL_EXAMPLES)} examples, {len(SCORERS)} scorers.")

Loaded 6 examples, 3 scorers.


## 4.  Baseline run

In [6]:
print("Running baseline...")
baseline_rows = run_eval(app, EVAL_EXAMPLES, SCORERS)
baseline = aggregate(baseline_rows)

for k, v in baseline.items():
    print(f"  baseline   {k:>16s}  =  {v:.3f}")

Running baseline...
  baseline     category_match  =  0.833
  baseline          format_ok  =  0.833
  baseline        helpfulness  =  0.967


## 5.  The optimization menu

Pick **ONE** quadrant. Apply ONE change. Re-run.

| Quadrant         | Examples                                                                      |
|------------------|-------------------------------------------------------------------------------|
| 🟦 Prompt-level    | Add few-shot examples to Classifier · stricter format in DRAFT_PROMPT          |
| 🟪 Workflow-level  | Skip QA when classify says urgency=low · cache retrieved docs per category     |
| 🟨 Model-level     | Cheaper model for Classifier/QA · keep Drafter on better model                 |
| 🟩 Reliability     | Retry on transient errors with `tenacity` · validate JSON before parsing       |

Below are four stubs — pick one and fill it in. Leave the others untouched.

### 🟦  Option A — Prompt-level (full implementation)

In [7]:
# 🟦  OPTION A — Prompt-level: few-shot examples in Classifier
#
# Three worked examples added — one per category — to anchor the model.

CLASSIFY_PROMPT_V2 = """\
Categorize the ticket as one of: billing, technical, account.
Rate urgency as: low, med, high.
Return JSON: {{"category": "...", "urgency": "..."}}

EXAMPLES:
TICKET: Why did my plan cost more this month?
{{"category": "billing", "urgency": "med"}}

TICKET: Login is broken and I have a demo in 10 minutes.
{{"category": "technical", "urgency": "high"}}

TICKET: How do I add a teammate as an admin?
{{"category": "account", "urgency": "low"}}

TICKET: {ticket}
"""


def classify_v2(state):
    r = llm.invoke(CLASSIFY_PROMPT_V2.format(ticket=state["ticket"]))
    p = parse_json(r.content)
    return {"category": p["category"], "urgency": p["urgency"]}

### 🟪  Option B — Workflow-level (full implementation)

In [8]:
# 🟪  OPTION B — Workflow-level: skip QA when urgency=low
#
# Add a conditional edge AFTER drafter that routes low-urgency tickets directly
# to END, bypassing the QA call entirely. High/med urgency still get reviewed.

def route_after_draft(state):
    """Conditional edge from drafter — skip QA for low-urgency tickets."""
    if state.get("urgency") == "low":
        return "END"
    return "qa"


def build_graph_v2_workflow(classify_fn=classify, retrieve_fn=retrieve,
                            draft_fn=draft, qa_fn=qa):
    """Variant graph: drafter → conditional(urgency) → qa | END."""
    g = StateGraph(TriageState)
    g.add_node("classify", classify_fn)
    g.add_node("retrieve", retrieve_fn)
    g.add_node("drafter",  draft_fn)
    g.add_node("qa",       qa_fn)
    g.set_entry_point("classify")
    g.add_edge("classify", "retrieve")
    g.add_edge("retrieve", "drafter")
    # CHANGED: was static drafter → qa. Now conditional on urgency.
    g.add_conditional_edges("drafter", route_after_draft,
        {"qa": "qa", "END": END})
    g.add_conditional_edges("qa", route_qa,
        {"drafter": "drafter", "END": END})
    return g.compile()


# Latency win for low-urgency tickets: ~1s of QA + drafter retry budget saved.
# Risk: quality regressions on low-urgency tickets won't be caught by QA.
# Mitigation: rely on the eval suite (Module 5) to surface those instead.

### 🟨  Option C — Model-level (full implementation)

In [9]:
# 🟨  OPTION C — Model-level: tier the models per node
#
# Classify + QA are simple decision tasks → cheap model.
# Drafter does the heavy generation → keep the better model (or upgrade).

llm_cheap   = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)   # classify + qa
llm_drafter = ChatOpenAI(model="gpt-4o-mini",   temperature=0)   # drafter (default)


def classify_v2_c(state):
    r = llm_cheap.invoke(CLASSIFY_PROMPT.format(ticket=state["ticket"]))
    p = parse_json(r.content)
    return {"category": p["category"], "urgency": p["urgency"]}


def draft_v2_c(state):
    ctx = "\n\n".join(f"[{d['id']}] {d['title']}\n{d['text']}"
                        for d in state["retrieved"])
    r = llm_drafter.invoke(DRAFT_PROMPT.format(context=ctx, ticket=state["ticket"]))
    return {"draft": r.content, "revisions": [r.content]}


def qa_v2_c(state):
    r = llm_cheap.invoke(QA_PROMPT.format(ticket=state["ticket"], draft=state["draft"]))
    p = parse_json(r.content)
    return {"verdict": p["verdict"], "revision_count": state["revision_count"] + 1}


# Cost: classify + qa typically cost ~half what drafter does. Moving them to
# gpt-3.5-turbo cuts that portion ~10x. End-to-end run cost typically drops 30-40%
# at minimal quality impact on these narrow decision tasks.
#
# Watch out for: model availability (gpt-3.5-turbo may not be on your tier),
# JSON format compliance (older models occasionally need stricter prompts).

### 🟩  Option D — Reliability-level (full implementation)

In [10]:
# 🟩  OPTION D — Reliability: retries with exponential backoff
#
# Wrap any node function with `with_retries` to make it resilient to transient
# LLM / parse failures.  Two attempts with a 1s gap is usually enough — the
# vast majority of OpenAI transients clear within seconds.

def with_retries(fn, attempts=3, base_delay=1.0):
    """Wrap a node function with exponential backoff + jitter.

    Re-raises the last exception if all attempts fail — so the graph still
    surfaces the error rather than silently producing bad output.
    """
    def wrapped(state):
        last_err = None
        for i in range(attempts):
            try:
                return fn(state)
            except Exception as e:
                last_err = e
                if i < attempts - 1:
                    delay = base_delay * (2 ** i) + random.uniform(0, 0.5)
                    time.sleep(delay)
        # All attempts failed — re-raise so caller sees the real error.
        raise last_err
    return wrapped


# Apply to every LLM-backed node. Retriever is pure dict lookup → no retry needed.
classify_v2_d = with_retries(classify)
draft_v2_d    = with_retries(draft)
qa_v2_d       = with_retries(qa)


# Production note: in a real system you'd also pair this with a
# 1) timeout on the LLM call itself (httpx timeout)
# 2) circuit breaker after N consecutive failures so you stop hammering a downed dep
# 3) tagged trace so retries are visible in observability (Module 4)

## 6.  Apply Option A and re-eval

*To measure a different option instead, comment out the Option A line and uncomment one of the others below.*

In [11]:
# === Pick ONE of these four lines, comment the rest ===

# Option A — prompt-level (few-shot in Classifier)
improved_app = build_graph(classify_fn=classify_v2)

# Option B — workflow-level (skip QA when urgency=low)
# improved_app = build_graph_v2_workflow()

# Option C — model-level (cheaper model for Classifier and QA)
# improved_app = build_graph(classify_fn=classify_v2_c,
#                            draft_fn=draft_v2_c, qa_fn=qa_v2_c)

# Option D — reliability-level (retries with exponential backoff)
# improved_app = build_graph(classify_fn=classify_v2_d,
#                            draft_fn=draft_v2_d, qa_fn=qa_v2_d)

print("Running improved candidate...")
improved_rows = run_eval(improved_app, EVAL_EXAMPLES, SCORERS)
improved = aggregate(improved_rows)

for k, v in improved.items():
    print(f"  improved  {k:>16s}  =  {v:.3f}")

Running improved candidate...
  improved    category_match  =  1.000
  improved         format_ok  =  0.833
  improved       helpfulness  =  0.967


## 7.  Compare before / after

In [12]:
print(f"  {'scorer':>16s}   baseline   improved   delta")
print("  " + "-" * 52)
for k in sorted(baseline):
    delta = improved[k] - baseline[k]
    mark = "✓" if delta >= 0 else "⚠"
    print(f"  {k:>16s}   {baseline[k]:.3f}     {improved[k]:.3f}    {delta:+.3f}  {mark}")

# Also count: how many examples flipped status on the helpfulness scorer?
def per_example_score(rows, key):
    return {r["example"]: r["score"] for r in rows if r["key"] == key}

base_help = per_example_score(baseline_rows, "helpfulness")
new_help  = per_example_score(improved_rows, "helpfulness")

print(f"\n  example-level helpfulness changes:")
for e in sorted(base_help):
    delta = new_help[e] - base_help[e]
    arrow = "▲" if delta > 0.05 else "▼" if delta < -0.05 else "—"
    print(f"    {e}:  {base_help[e]:.2f}  →  {new_help[e]:.2f}   {arrow}")

            scorer   baseline   improved   delta
  ----------------------------------------------------
    category_match   0.833     1.000    +0.167  ✓
         format_ok   0.833     0.833    +0.000  ✓
       helpfulness   0.967     0.967    +0.000  ✓

  example-level helpfulness changes:
    E-001:  1.00  →  1.00   —
    E-002:  1.00  →  1.00   —
    E-003:  1.00  →  1.00   —
    E-004:  1.00  →  1.00   —
    E-005:  1.00  →  1.00   —
    E-006:  0.80  →  0.80   —


## 8.  Write up your change

Fill in the dict below. This is the *real* deliverable of the capstone — a small artifact you can paste into a PR description tomorrow.

In [13]:
CAPSTONE = {
    "quadrant":     "prompt",
    "change":       "Added 3 few-shot examples (one per category) to CLASSIFY_PROMPT.",
    "metric_moved": "category_match",
    "delta":        0.17,   # actual number will depend on your run
    "regressed_anything": False,
    "would_ship":   True,
    "why": (
        "Few-shot anchoring is the cheapest prompt-level win for classification "
        "tasks. The cost (~50 extra prompt tokens per classify call) is negligible "
        "compared to the accuracy lift, and no other scorer regressed."
    ),
}
print(CAPSTONE)

{'quadrant': 'prompt', 'change': 'Added 3 few-shot examples (one per category) to CLASSIFY_PROMPT.', 'metric_moved': 'category_match', 'delta': 0.17, 'regressed_anything': False, 'would_ship': True, 'why': 'Few-shot anchoring is the cheapest prompt-level win for classification tasks. The cost (~50 extra prompt tokens per classify call) is negligible compared to the accuracy lift, and no other scorer regressed.'}


## Wrap up — and that's the workshop

You came in able to *prototype* an agent. You're leaving able to *ship* one.

Recap of what's in your toolkit now:

1. **Design** — typed shared state, explicit topology, termination guards
2. **Build** — LangGraph primitives, the four-layer agent anatomy
3. **Debug** — three failure families + the reproduce-isolate-fix loop
4. **Observe** — traces, metadata, the bottleneck-finding workflow
5. **Evaluate** — rule-based + LLM-as-judge scorers, regression catching
6. **Optimize** — pick one quadrant, measure, repeat

Take this notebook home. Swap in your own knowledge base. Swap in your own use case. Ship it.

Thank you for spending the day.